In [49]:
"""
Evapotranspiration Prediction - Ensemble Model v4
INSIGHT: ET is cumulative & Days_Planted is the strongest driver (-0.40 on daily).
Strategy: Predict RAW cumulative ET directly. Include Days_Planted prominently.
Add polynomial + interaction features to boost signal.
Data Source: TOMATO.xlsx
"""

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor,
    ExtraTreesRegressor, VotingRegressor, StackingRegressor
)
from sklearn.linear_model import Ridge, ElasticNet, BayesianRidge
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, RobustScaler, QuantileTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import json

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False



In [50]:
print("=" * 65)
print("   EVAPOTRANSPIRATION ENSEMBLE MODEL  v4")
print("   Cumulative ET Prediction — Tomato Crop")
print("=" * 65)

FILE_PATH = "TOMATO.xlsx"

try:
    df = pd.read_excel(FILE_PATH)
    print(f"\n✅ Loaded: {FILE_PATH}  |  Shape: {df.shape}")
except FileNotFoundError:
    raise FileNotFoundError(
        f"\n❌ '{FILE_PATH}' not found.\n"
        "Set FILE_PATH to the full path of your file."
    )



   EVAPOTRANSPIRATION ENSEMBLE MODEL  v4
   Cumulative ET Prediction — Tomato Crop

✅ Loaded: TOMATO.xlsx  |  Shape: (3000, 13)


In [51]:
COLUMN_MAP = {
    "Temperature_C":      "Temperature [_ C]",
    "Humidity_pct":       "Humidity [%]",
    "Soil_Moisture":      "Soil moisture",
    "Crop_Coefficient":   "Crop Coefficient",
    "Nitrogen":           "Nitrogen [mg/kg]",
    "Phosphorus":         "Phosphorus [mg/kg]",
    "Potassium":          "Potassium",
    "Wind_Speed":         "Wind Speed",
    "Days_Planted":       "Days of planted",
    "pH":                 "pH",
    "Evapotranspiration": "Evapotranspiration",
}
# Note: Reference evapotranspiration removed (near-zero correlation with target)

df = df.rename(columns={v: k for k, v in COLUMN_MAP.items()})
missing = [c for c in COLUMN_MAP if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}\nUpdate COLUMN_MAP to match your headers.")

df = df[list(COLUMN_MAP.keys())]
for c in df.columns:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df = df.dropna(subset=['Evapotranspiration'])
df = df[df['Days_Planted'] > 0]
df = df.fillna(df.median(numeric_only=True))
print(f"   Rows after cleaning: {len(df)}")



   Rows after cleaning: 3000


In [52]:
print("\n" + "─"*65)
print("CORRELATION CHECK  (raw features vs Evapotranspiration)")
print("─"*65)
corr = df.corr(numeric_only=True)['Evapotranspiration'].drop('Evapotranspiration').sort_values(key=abs, ascending=False)
for feat, c in corr.items():
    bar = "█" * int(abs(c) * 40)
    sign = "+" if c >= 0 else "-"
    print(f"  {feat:<25} {sign}{abs(c):.4f}  {bar}")



─────────────────────────────────────────────────────────────────
CORRELATION CHECK  (raw features vs Evapotranspiration)
─────────────────────────────────────────────────────────────────
  Days_Planted              +0.2588  ██████████
  Crop_Coefficient          +0.2231  ████████
  Phosphorus                +0.1917  ███████
  Potassium                 +0.1369  █████
  Wind_Speed                +0.1247  ████
  Soil_Moisture             +0.1109  ████
  Temperature_C             -0.1072  ████
  Nitrogen                  -0.0838  ███
  pH                        +0.0668  ██
  Humidity_pct              -0.0369  █


In [53]:
print("\n" + "─"*65)
print("FEATURE ENGINEERING")
print("─"*65)

# Time-based features (most important)
df['Days_sq']             = df['Days_Planted'] ** 2
df['Days_sqrt']           = np.sqrt(df['Days_Planted'])
df['Days_log']            = np.log1p(df['Days_Planted'])
df['Growth_Stage']        = pd.cut(df['Days_Planted'],
                                    bins=[0,20,40,70,100,130,999],
                                    labels=[1,2,3,4,5,6]).astype(float).fillna(3)

# Crop & time features
df['Kc_days']             = df['Crop_Coefficient'] * df['Days_Planted']  # ★ cumulative proxy

# Atmospheric demand
df['VPD']                 = (1 - df['Humidity_pct']/100) * 0.6108 * \
                             np.exp(17.27 * df['Temperature_C'] / (df['Temperature_C'] + 237.3))
df['VPD_days']            = df['VPD'] * df['Days_Planted']          # ★ cumulative proxy
df['VPD_Wind']            = df['VPD'] * df['Wind_Speed']

# Temperature features
df['Temp_sq']             = df['Temperature_C'] ** 2
df['Temp_days']           = df['Temperature_C'] * df['Days_Planted']
df['Temp_Humidity']       = df['Temperature_C'] / (df['Humidity_pct'] + 1)

# Soil & stress
df['Stress_Index']        = df['Soil_Moisture'] * df['Humidity_pct'] / 100
df['Soil_Temp']           = df['Soil_Moisture'] * df['Temperature_C']
df['NPK_Total']           = df['Nitrogen'] + df['Phosphorus'] + df['Potassium']
df['Wind_Temp']           = df['Wind_Speed'] * df['Temperature_C']

# Log transforms
df['log_Kc_days']         = np.log1p(df['Kc_days'])
df['log_Days']            = np.log1p(df['Days_Planted'])

FEATURES = [
    # Core inputs (Ref_ET removed — near-zero correlation with target)
    'Temperature_C', 'Humidity_pct', 'Soil_Moisture',
    'Crop_Coefficient', 'Nitrogen', 'Phosphorus', 'Potassium',
    'Wind_Speed', 'Days_Planted', 'pH',
    # Time features ★
    'Days_sq', 'Days_sqrt', 'Days_log', 'Growth_Stage',
    # Cumulative proxies ★★ (most predictive)
    'Kc_days', 'VPD_days', 'Temp_days',
    # Atmospheric
    'VPD', 'VPD_Wind',
    # Temp
    'Temp_sq', 'Temp_Humidity',
    # Soil & stress
    'Stress_Index', 'Soil_Temp', 'NPK_Total', 'Wind_Temp',
    # Log transforms
    'log_Kc_days', 'log_Days',
]



─────────────────────────────────────────────────────────────────
FEATURE ENGINEERING
─────────────────────────────────────────────────────────────────


In [54]:
TARGET = 'Evapotranspiration'   # predict cumulative ET directly

# Quick correlation check on engineered features
eng_corr = df[FEATURES + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
print(f"\n  Top 10 engineered feature correlations with ET:")
for feat, c in eng_corr.head(10).items():
    bar = "█" * int(abs(c) * 40)
    sign = "+" if c >= 0 else "-"
    print(f"    {feat:<25} {sign}{abs(c):.4f}  {bar}")




  Top 10 engineered feature correlations with ET:
    log_Kc_days               +0.3140  ████████████
    Days_log                  +0.3101  ████████████
    log_Days                  +0.3101  ████████████
    Days_sqrt                 +0.2932  ███████████
    Growth_Stage              +0.2635  ██████████
    Days_Planted              +0.2588  ██████████
    Kc_days                   +0.2336  █████████
    Crop_Coefficient          +0.2231  ████████
    Temp_days                 +0.2221  ████████
    Days_sq                   +0.1925  ███████


In [55]:
Q1 = df[TARGET].quantile(0.01)
Q3 = df[TARGET].quantile(0.99)
before = len(df)
df = df[(df[TARGET] >= Q1) & (df[TARGET] <= Q3)]
print(f"\n  Outlier removal (1–99%): {before} → {len(df)} rows")

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"  Train: {len(X_train)} | Test: {len(X_test)}")
print(f"  Target → min={y.min():.1f}  max={y.max():.1f}  mean={y.mean():.1f}")



  Outlier removal (1–99%): 3000 → 2940 rows
  Train: 2352 | Test: 588
  Target → min=5.5  max=888.9  mean=415.0


In [56]:
base_models = {
    'Random Forest': RandomForestRegressor(
        n_estimators=500, max_depth=15, min_samples_split=3,
        n_jobs=-1, random_state=42),
    'Extra Trees': ExtraTreesRegressor(
        n_estimators=500, max_depth=15, n_jobs=-1, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=500, learning_rate=0.03, max_depth=6,
        subsample=0.8, min_samples_split=4, random_state=42),
    'Ridge': Pipeline([
        ('scaler', RobustScaler()), ('model', Ridge(alpha=0.5))]),
    'ElasticNet': Pipeline([
        ('scaler', RobustScaler()),
        ('model', ElasticNet(alpha=0.01, l1_ratio=0.3, max_iter=3000))]),
    'SVR': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVR(C=50, epsilon=0.5, kernel='rbf'))]),
    'BayesianRidge': Pipeline([
        ('scaler', StandardScaler()), ('model', BayesianRidge())]),
}

if XGBOOST_AVAILABLE:
    base_models['XGBoost'] = XGBRegressor(
        n_estimators=500, learning_rate=0.03, max_depth=7,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        n_jobs=-1, random_state=42, verbosity=0)

if LGBM_AVAILABLE:
    base_models['LightGBM'] = LGBMRegressor(
        n_estimators=500, learning_rate=0.03, max_depth=7,
        num_leaves=80, min_child_samples=10,
        n_jobs=-1, random_state=42, verbose=-1)


In [57]:
print("\n" + "─"*65)
print("INDIVIDUAL MODEL PERFORMANCE  (target: Cumulative ET)")
print("─"*65)

results = {}
for name, model in base_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    cv   = cross_val_score(model, X_train, y_train, cv=5,
                           scoring='r2', n_jobs=-1).mean()
    results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'CV_R2': cv}
    status = "✅" if r2 > 0.7 else ("⚠️ " if r2 > 0.4 else "❌")
    print(f"  {status} {name:<20} | RMSE={rmse:.2f} | MAE={mae:.2f} | R²={r2:.4f} | CV_R²={cv:.4f}")



─────────────────────────────────────────────────────────────────
INDIVIDUAL MODEL PERFORMANCE  (target: Cumulative ET)
─────────────────────────────────────────────────────────────────
  ❌ Random Forest        | RMSE=256.75 | MAE=215.52 | R²=0.1299 | CV_R²=0.0963
  ❌ Extra Trees          | RMSE=256.46 | MAE=215.07 | R²=0.1319 | CV_R²=0.0930
  ❌ Gradient Boosting    | RMSE=264.87 | MAE=219.83 | R²=0.0740 | CV_R²=0.0057
  ❌ Ridge                | RMSE=258.66 | MAE=223.87 | R²=0.1169 | CV_R²=0.1068
  ❌ ElasticNet           | RMSE=258.89 | MAE=225.21 | R²=0.1153 | CV_R²=0.1047
  ❌ SVR                  | RMSE=258.03 | MAE=220.20 | R²=0.1212 | CV_R²=0.0897
  ❌ BayesianRidge        | RMSE=259.80 | MAE=226.64 | R²=0.1091 | CV_R²=0.0989
  ❌ XGBoost              | RMSE=265.59 | MAE=222.57 | R²=0.0690 | CV_R²=0.0038
  ❌ LightGBM             | RMSE=264.99 | MAE=220.02 | R²=0.0732 | CV_R²=-0.0324


In [58]:
print("\n" + "─"*65)
print("ENSEMBLE MODELS")
print("─"*65)

top_models = sorted(results.items(), key=lambda x: x[1]['R2'], reverse=True)[:5]
print(f"\n  Top 5: {[m[0] for m in top_models]}")

voting_estimators = [(name, base_models[name]) for name, _ in top_models]
weights = [max(m[1]['R2'], 0.01) for m in top_models]

voting_ensemble = VotingRegressor(estimators=voting_estimators, weights=weights, n_jobs=-1)
voting_ensemble.fit(X_train, y_train)
v_pred = voting_ensemble.predict(X_test)
v_r2   = r2_score(y_test, v_pred)
v_rmse = np.sqrt(mean_squared_error(y_test, v_pred))
print(f"  Voting Ensemble    | RMSE={v_rmse:.2f} | R²={v_r2:.4f}")

stacking_ensemble = StackingRegressor(
    estimators=voting_estimators[:4],
    final_estimator=Ridge(alpha=0.5),
    cv=5, n_jobs=-1
)
stacking_ensemble.fit(X_train, y_train)
s_pred = stacking_ensemble.predict(X_test)
s_r2   = r2_score(y_test, s_pred)
s_rmse = np.sqrt(mean_squared_error(y_test, s_pred))
print(f"  Stacking Ensemble  | RMSE={s_rmse:.2f} | R²={s_r2:.4f}")

if s_r2 >= v_r2:
    best_model, best_name = stacking_ensemble, "Stacking Ensemble"
    best_r2, best_rmse, best_pred = s_r2, s_rmse, s_pred
else:
    best_model, best_name = voting_ensemble, "Voting Ensemble"
    best_r2, best_rmse, best_pred = v_r2, v_rmse, v_pred

print(f"\n  ★ BEST: {best_name} | R²={best_r2:.4f} | RMSE={best_rmse:.2f}")




─────────────────────────────────────────────────────────────────
ENSEMBLE MODELS
─────────────────────────────────────────────────────────────────

  Top 5: ['Extra Trees', 'Random Forest', 'SVR', 'Ridge', 'ElasticNet']
  Voting Ensemble    | RMSE=256.11 | R²=0.1342
  Stacking Ensemble  | RMSE=256.89 | R²=0.1290

  ★ BEST: Voting Ensemble | R²=0.1342 | RMSE=256.11


In [59]:
best_tree = base_models[top_models[0][0]]
if hasattr(best_tree, 'feature_importances_'):
    fi = pd.Series(best_tree.feature_importances_, index=FEATURES).sort_values(ascending=False)
    print("\n" + "─"*65)
    print(f"TOP 12 FEATURE IMPORTANCES  ({top_models[0][0]})")
    print("─"*65)
    for feat, imp in fi.head(12).items():
        bar = "█" * int(imp * 300)
        print(f"  {feat:<25} {imp:.4f}  {bar}")


─────────────────────────────────────────────────────────────────
TOP 12 FEATURE IMPORTANCES  (Extra Trees)
─────────────────────────────────────────────────────────────────
  log_Kc_days               0.0649  ███████████████████
  Kc_days                   0.0528  ███████████████
  Crop_Coefficient          0.0525  ███████████████
  Phosphorus                0.0458  █████████████
  Days_log                  0.0440  █████████████
  pH                        0.0432  ████████████
  NPK_Total                 0.0405  ████████████
  Days_sqrt                 0.0405  ████████████
  Nitrogen                  0.0394  ███████████
  Potassium                 0.0389  ███████████
  Wind_Speed                0.0365  ██████████
  log_Days                  0.0358  ██████████


In [60]:
def predict_water_requirement(model, input_dict, area_hectares=1.0):
    """
    Predict cumulative ET (mm) and convert to water volume.
    Daily ET  = cumulative ET / Days_Planted
    Water m³  = Daily ET × area × 10
    """
    row = pd.DataFrame([input_dict])

    # Feature engineering (must match training)
    row['Days_sq']         = row['Days_Planted'] ** 2
    row['Days_sqrt']       = np.sqrt(row['Days_Planted'])
    row['Days_log']        = np.log1p(row['Days_Planted'])
    row['Growth_Stage']    = float(pd.cut([row['Days_Planted'].values[0]],
                                           bins=[0,20,40,70,100,130,999],
                                           labels=[1,2,3,4,5,6])[0])
    row['Kc_days']         = row['Crop_Coefficient'] * row['Days_Planted']
    row['VPD']             = (1 - row['Humidity_pct']/100) * 0.6108 * \
                              np.exp(17.27*row['Temperature_C']/(row['Temperature_C']+237.3))
    row['VPD_days']        = row['VPD'] * row['Days_Planted']
    row['VPD_Wind']        = row['VPD'] * row['Wind_Speed']
    row['Temp_days']       = row['Temperature_C'] * row['Days_Planted']
    row['Temp_sq']         = row['Temperature_C'] ** 2
    row['Temp_Humidity']   = row['Temperature_C'] / (row['Humidity_pct'] + 1)
    row['Stress_Index']    = row['Soil_Moisture'] * row['Humidity_pct'] / 100
    row['Soil_Temp']       = row['Soil_Moisture'] * row['Temperature_C']
    row['NPK_Total']       = row['Nitrogen'] + row['Phosphorus'] + row['Potassium']
    row['Wind_Temp']       = row['Wind_Speed'] * row['Temperature_C']
    row['log_Kc_days']     = np.log1p(row['Kc_days'])
    row['log_Days']        = np.log1p(row['Days_Planted'])

    et_cumulative = max(model.predict(row[FEATURES])[0], 0)
    days          = max(input_dict['Days_Planted'], 1)
    et_daily      = et_cumulative / days

    water_day_m3  = et_daily * area_hectares * 10
    water_day_L   = water_day_m3 * 1000
    water_total_m3= et_cumulative * area_hectares * 10

    if et_daily < 2:
        urgency, advice = "Low",      "🟢 LOW — Minimal irrigation needed today."
    elif et_daily < 5:
        urgency, advice = "Moderate", "🟡 MODERATE — Irrigate regularly."
    elif et_daily < 8:
        urgency, advice = "High",     "🟠 HIGH — Increase irrigation frequency."
    else:
        urgency, advice = "Critical", "🔴 CRITICAL — Irrigate immediately!"

    return {
        'ET_cumulative_mm':     round(et_cumulative, 1),
        'ET_daily_mm':          round(et_daily, 3),
        'Water_today_m3_ha':    round(water_day_m3, 2),
        'Water_today_L_ha':     round(water_day_L, 0),
        'Water_season_m3_ha':   round(water_total_m3, 1),
        'Urgency':              urgency,
        'Advice':               advice,
    }



In [61]:
print("\n" + "─"*65)
print("FARMER PREDICTIONS  (1 hectare)")
print("─"*65)

base_cols = ['Temperature_C','Humidity_pct','Soil_Moisture',
             'Crop_Coefficient','Nitrogen','Phosphorus','Potassium',
             'Wind_Speed','Days_Planted','pH']
mean_vals = df[base_cols].mean().to_dict()

scenarios = {
    "Avg Conditions (Day {:.0f})".format(mean_vals['Days_Planted']): mean_vals,
    "Early Stage    (Day 20)":   {**mean_vals, 'Days_Planted': 20,  'Crop_Coefficient': 0.6},
    "Flowering      (Day 60)":   {**mean_vals, 'Days_Planted': 60,  'Crop_Coefficient': 1.15},
    "Peak Season    (Day 90)":   {**mean_vals, 'Days_Planted': 90,  'Crop_Coefficient': 1.2},
    "Hot Dry Stress (Day 60)":   {**mean_vals, 'Days_Planted': 60,
                                   'Temperature_C': df['Temperature_C'].max(),
                                   'Humidity_pct':  df['Humidity_pct'].min()},
}

for name, data in scenarios.items():
    r = predict_water_requirement(best_model, data, area_hectares=1.0)
    print(f"\n  📍 {name}")
    print(f"     Cumul. ET      : {r['ET_cumulative_mm']} mm total")
    print(f"     Daily ET       : {r['ET_daily_mm']} mm/day")
    print(f"     Water TODAY    : {r['Water_today_m3_ha']} m³/ha  ({r['Water_today_L_ha']:,.0f} L/ha)")
    print(f"     Season Total   : {r['Water_season_m3_ha']} m³/ha")
    print(f"     Urgency        : {r['Urgency']}")
    print(f"     {r['Advice']}")




─────────────────────────────────────────────────────────────────
FARMER PREDICTIONS  (1 hectare)
─────────────────────────────────────────────────────────────────

  📍 Avg Conditions (Day 80)
     Cumul. ET      : 446.0 mm total
     Daily ET       : 5.564 mm/day
     Water TODAY    : 55.64 m³/ha  (55,640 L/ha)
     Season Total   : 4460.1 m³/ha
     Urgency        : High
     🟠 HIGH — Increase irrigation frequency.

  📍 Early Stage    (Day 20)
     Cumul. ET      : 172.0 mm total
     Daily ET       : 8.599 mm/day
     Water TODAY    : 85.99 m³/ha  (85,985 L/ha)
     Season Total   : 1719.7 m³/ha
     Urgency        : Critical
     🔴 CRITICAL — Irrigate immediately!

  📍 Flowering      (Day 60)
     Cumul. ET      : 436.6 mm total
     Daily ET       : 7.276 mm/day
     Water TODAY    : 72.76 m³/ha  (72,759 L/ha)
     Season Total   : 4365.5 m³/ha
     Urgency        : High
     🟠 HIGH — Increase irrigation frequency.

  📍 Peak Season    (Day 90)
     Cumul. ET      : 440.8 mm total

In [2]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor,
    ExtraTreesRegressor, VotingRegressor, StackingRegressor
)
from sklearn.linear_model import Ridge, ElasticNet, BayesianRidge
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import json

In [3]:
try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False

In [4]:
print("=" * 60)
print("   EVAPOTRANSPIRATION ENSEMBLE MODEL")
print("   Water Requirement Prediction for Tomato Farmers")
print("=" * 60)

FILE_PATH = "TOMATO.xlsx" 

try:
    df = pd.read_excel(FILE_PATH)
    print(f"\n✅ Loaded: {FILE_PATH}")
except FileNotFoundError:
    raise FileNotFoundError(
        f"\n❌ File not found: '{FILE_PATH}'\n"
        "Please update FILE_PATH to the full path of your TOMATO.xlsx file.\n"
        "Example: FILE_PATH = 'C:/Users/YourName/Desktop/TOMATO.xlsx'"
    )

   EVAPOTRANSPIRATION ENSEMBLE MODEL
   Water Requirement Prediction for Tomato Farmers

✅ Loaded: TOMATO.xlsx


In [5]:
print(f"   Shape: {df.shape}")
print(f"\n   Columns detected:\n   {list(df.columns)}")

   Shape: (3000, 13)

   Columns detected:
   ['Temperature [_ C]', 'Humidity [%]', 'Soil moisture', 'Reference evapotranspiration', 'Evapotranspiration', 'Crop Coefficient', 'Nitrogen [mg/kg]', 'Phosphorus [mg/kg]', 'Potassium', 'Wind Speed', 'Days of planted', 'pH', 'Unnamed: 12']


In [6]:
COLUMN_MAP = {
    "Temperature_C":      "Temperature [_ C]",
    "Humidity_pct":       "Humidity [%]",
    "Soil_Moisture":      "Soil moisture",
    "Ref_ET":             "Reference evapotranspiration",
    "Crop_Coefficient":   "Crop Coefficient",
    "Nitrogen":           "Nitrogen [mg/kg]",
    "Phosphorus":         "Phosphorus [mg/kg]",
    "Potassium":          "Potassium",
    "Wind_Speed":         "Wind Speed",
    "Days_Planted":       "Days of planted",
    "pH":                 "pH",
    "Evapotranspiration": "Evapotranspiration",
}

In [7]:
df = df.rename(columns={v: k for k, v in COLUMN_MAP.items()})

# Verify all required columns exist
missing = [c for c in COLUMN_MAP.keys() if c not in df.columns]
if missing:
    print(f"\n⚠️  Missing columns after mapping: {missing}")
    print("   Please update COLUMN_MAP above to match your Excel headers exactly.")
    raise ValueError(f"Missing columns: {missing}")

df = df[list(COLUMN_MAP.keys())]

In [8]:
df['Temp_Humidity_ratio'] = df['Temperature_C'] / (df['Humidity_pct'] + 1)
df['ET_Kc']               = df['Ref_ET'] * df['Crop_Coefficient']
df['VPD']                 = (1 - df['Humidity_pct']/100) * 0.6108 * \
                             np.exp(17.27 * df['Temperature_C'] / (df['Temperature_C'] + 237.3))
df['Stress_Index']        = df['Soil_Moisture'] * df['Humidity_pct'] / 100
df['Growth_Stage']        = pd.cut(df['Days_Planted'],
                                    bins=[0,30,80,130,180,999],
                                    labels=[1,2,3,4,5]).astype(float).fillna(3)
df['Wind_Temp']           = df['Wind_Speed'] * df['Temperature_C']
df['NPK_Total']           = df['Nitrogen'] + df['Phosphorus'] + df['Potassium']

FEATURES = [
    'Temperature_C','Humidity_pct','Soil_Moisture','Ref_ET',
    'Crop_Coefficient','Nitrogen','Phosphorus','Potassium',
    'Wind_Speed','Days_Planted','pH',
    'Temp_Humidity_ratio','ET_Kc','VPD','Stress_Index',
    'Growth_Stage','Wind_Temp','NPK_Total'
]

In [9]:
TARGET = 'Evapotranspiration'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [10]:
print(f"\n   Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")


   Train: 2400 rows | Test: 600 rows


In [11]:
base_models = {
    'Random Forest': RandomForestRegressor(
        n_estimators=300, max_depth=12, min_samples_split=4,
        n_jobs=-1, random_state=42),
    'Extra Trees': ExtraTreesRegressor(
        n_estimators=300, max_depth=12, n_jobs=-1, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=5,
        subsample=0.8, random_state=42),
    'Ridge Regression': Pipeline([
        ('scaler', RobustScaler()), ('model', Ridge(alpha=1.0))]),
    'ElasticNet': Pipeline([
        ('scaler', RobustScaler()),
        ('model', ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=2000))]),
    'SVR': Pipeline([
        ('scaler', StandardScaler()), ('model', SVR(C=10, epsilon=0.1))]),
    'BayesianRidge': Pipeline([
        ('scaler', StandardScaler()), ('model', BayesianRidge())]),
}

In [12]:
if XGBOOST_AVAILABLE:
    base_models['XGBoost'] = XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        n_jobs=-1, random_state=42, verbosity=0)

if LGBM_AVAILABLE:
    base_models['LightGBM'] = LGBMRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        num_leaves=63, n_jobs=-1, random_state=42, verbose=-1)

In [13]:
print("\n" + "─"*60)
print("INDIVIDUAL MODEL PERFORMANCE")
print("─"*60)

results = {}
for name, model in base_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    cv   = cross_val_score(model, X_train, y_train, cv=3,
                           scoring='r2', n_jobs=-1).mean()
    results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'CV_R2': cv}
    print(f"  {name:<20} | RMSE={rmse:.4f} | MAE={mae:.4f} | R²={r2:.4f} | CV_R²={cv:.4f}")


────────────────────────────────────────────────────────────
INDIVIDUAL MODEL PERFORMANCE
────────────────────────────────────────────────────────────
  Random Forest        | RMSE=251.5509 | MAE=207.9806 | R²=0.1517 | CV_R²=0.1105
  Extra Trees          | RMSE=250.9918 | MAE=207.3856 | R²=0.1555 | CV_R²=0.1150
  Gradient Boosting    | RMSE=256.7930 | MAE=211.1315 | R²=0.1160 | CV_R²=0.0195
  Ridge Regression     | RMSE=253.4611 | MAE=214.7078 | R²=0.1388 | CV_R²=0.1008
  ElasticNet           | RMSE=254.7778 | MAE=217.6708 | R²=0.1298 | CV_R²=0.0995
  SVR                  | RMSE=252.2259 | MAE=211.2635 | R²=0.1472 | CV_R²=0.1116
  BayesianRidge        | RMSE=254.8609 | MAE=217.8469 | R²=0.1293 | CV_R²=0.0912
  XGBoost              | RMSE=260.7215 | MAE=213.3728 | R²=0.0888 | CV_R²=0.0175
  LightGBM             | RMSE=257.6161 | MAE=211.0789 | R²=0.1103 | CV_R²=-0.0021


In [14]:
print("\n" + "─"*60)
print("ENSEMBLE MODELS")
print("─"*60)

top_models = sorted(results.items(), key=lambda x: x[1]['R2'], reverse=True)[:5]
print(f"\nTop 5 for ensemble: {[m[0] for m in top_models]}")

voting_estimators = [(name, base_models[name]) for name, _ in top_models]
weights = [m[1]['R2'] for m in top_models]

voting_ensemble = VotingRegressor(estimators=voting_estimators, weights=weights, n_jobs=-1)
voting_ensemble.fit(X_train, y_train)
y_pred_vote = voting_ensemble.predict(X_test)
v_r2   = r2_score(y_test, y_pred_vote)
v_rmse = np.sqrt(mean_squared_error(y_test, y_pred_vote))
print(f"  Voting Ensemble       | RMSE={v_rmse:.4f} | R²={v_r2:.4f}")

stacking_ensemble = StackingRegressor(
    estimators=voting_estimators[:4],
    final_estimator=Ridge(alpha=1.0),
    cv=5, n_jobs=-1
)
stacking_ensemble.fit(X_train, y_train)
y_pred_stack = stacking_ensemble.predict(X_test)
s_r2   = r2_score(y_test, y_pred_stack)
s_rmse = np.sqrt(mean_squared_error(y_test, y_pred_stack))
print(f"  Stacking Ensemble     | RMSE={s_rmse:.4f} | R²={s_r2:.4f}")

if s_r2 >= v_r2:
    best_model, best_name = stacking_ensemble, "Stacking Ensemble"
    best_r2, best_rmse = s_r2, s_rmse
else:
    best_model, best_name = voting_ensemble, "Voting Ensemble"
    best_r2, best_rmse = v_r2, v_rmse

print(f"\n★ BEST MODEL: {best_name} | R²={best_r2:.4f} | RMSE={best_rmse:.4f}")


────────────────────────────────────────────────────────────
ENSEMBLE MODELS
────────────────────────────────────────────────────────────

Top 5 for ensemble: ['Extra Trees', 'Random Forest', 'SVR', 'Ridge Regression', 'ElasticNet']
  Voting Ensemble       | RMSE=250.9354 | R²=0.1559
  Stacking Ensemble     | RMSE=251.1461 | R²=0.1545

★ BEST MODEL: Voting Ensemble | R²=0.1559 | RMSE=250.9354


In [15]:
best_tree = base_models[top_models[0][0]]
if hasattr(best_tree, 'feature_importances_'):
    fi = pd.Series(best_tree.feature_importances_, index=FEATURES).sort_values(ascending=False)
    print("\n" + "─"*60)
    print(f"TOP 10 FEATURE IMPORTANCES (from {top_models[0][0]})")
    print("─"*60)
    for feat, imp in fi.head(10).items():
        bar = "█" * int(imp * 200)
        print(f"  {feat:<25} {imp:.4f}  {bar}")


────────────────────────────────────────────────────────────
TOP 10 FEATURE IMPORTANCES (from Extra Trees)
────────────────────────────────────────────────────────────
  Growth_Stage              0.1601  ████████████████████████████████
  Days_Planted              0.1058  █████████████████████
  Crop_Coefficient          0.0908  ██████████████████
  Phosphorus                0.0548  ██████████
  ET_Kc                     0.0521  ██████████
  Ref_ET                    0.0520  ██████████
  pH                        0.0458  █████████
  NPK_Total                 0.0445  ████████
  Nitrogen                  0.0443  ████████
  Potassium                 0.0436  ████████


In [16]:
def predict_water_requirement(model, features_dict, area_hectares=1.0):
    row = pd.DataFrame([features_dict])
    row['Temp_Humidity_ratio'] = row['Temperature_C'] / (row['Humidity_pct'] + 1)
    row['ET_Kc']               = row['Ref_ET'] * row['Crop_Coefficient']
    row['VPD']                 = (1 - row['Humidity_pct']/100) * 0.6108 * \
                                  np.exp(17.27*row['Temperature_C']/(row['Temperature_C']+237.3))
    row['Stress_Index']        = row['Soil_Moisture'] * row['Humidity_pct'] / 100
    row['Growth_Stage']        = float(pd.cut([row['Days_Planted'].values[0]],
                                               bins=[0,30,80,130,180,999],
                                               labels=[1,2,3,4,5])[0])
    row['Wind_Temp']           = row['Wind_Speed'] * row['Temperature_C']
    row['NPK_Total']           = row['Nitrogen'] + row['Phosphorus'] + row['Potassium']

    et_pred  = max(model.predict(row[FEATURES])[0], 0)
    water_m3 = et_pred * area_hectares * 10
    water_L  = water_m3 * 1000

    if et_pred < 2:
        urgency, advice = "Low",      "🟢 LOW — Minimal irrigation needed."
    elif et_pred < 5:
        urgency, advice = "Moderate", "🟡 MODERATE — Regular irrigation recommended."
    elif et_pred < 8:
        urgency, advice = "High",     "🟠 HIGH — Increase irrigation frequency."
    else:
        urgency, advice = "Critical", "🔴 CRITICAL — Immediate heavy irrigation required!"

    return {
        'ET_mm_per_day':       round(et_pred, 3),
        'Water_m3_per_ha':     round(water_m3, 1),
        'Water_Liters_per_ha': round(water_L, 0),
        'Urgency':             urgency,
        'Advice':              advice
    }

In [17]:
print("\n" + "─"*60)
print("SAMPLE PREDICTIONS (based on your data)")
print("─"*60)

base_cols = ['Temperature_C','Humidity_pct','Soil_Moisture','Ref_ET',
             'Crop_Coefficient','Nitrogen','Phosphorus','Potassium',
             'Wind_Speed','Days_Planted','pH']
mean_vals = df[base_cols].mean().to_dict()

scenarios = {
    "Avg Conditions":  mean_vals,
    "Hot Dry Stress":  {**mean_vals, 'Temperature_C': df['Temperature_C'].max(),
                                      'Humidity_pct':  df['Humidity_pct'].min()},
    "Cool Humid":      {**mean_vals, 'Temperature_C': df['Temperature_C'].min(),
                                      'Humidity_pct':  df['Humidity_pct'].max()},
}

for name, data in scenarios.items():
    r = predict_water_requirement(best_model, data, area_hectares=1.0)
    print(f"\n  📍 {name}")
    print(f"     ET Predicted  : {r['ET_mm_per_day']} mm/day")
    print(f"     Water Needed  : {r['Water_m3_per_ha']} m³/ha/day  ({r['Water_Liters_per_ha']:,.0f} L/ha/day)")
    print(f"     Urgency       : {r['Urgency']}")
    print(f"     {r['Advice']}")



────────────────────────────────────────────────────────────
SAMPLE PREDICTIONS (based on your data)
────────────────────────────────────────────────────────────

  📍 Avg Conditions
     ET Predicted  : 452.084 mm/day
     Water Needed  : 4520.8 m³/ha/day  (4,520,837 L/ha/day)
     Urgency       : Critical
     🔴 CRITICAL — Immediate heavy irrigation required!

  📍 Hot Dry Stress
     ET Predicted  : 402.508 mm/day
     Water Needed  : 4025.1 m³/ha/day  (4,025,079 L/ha/day)
     Urgency       : Critical
     🔴 CRITICAL — Immediate heavy irrigation required!

  📍 Cool Humid
     ET Predicted  : 450.227 mm/day
     Water Needed  : 4502.3 m³/ha/day  (4,502,266 L/ha/day)
     Urgency       : Critical
     🔴 CRITICAL — Immediate heavy irrigation required!


In [18]:
joblib.dump(best_model, 'et_ensemble_model.pkl')
joblib.dump(FEATURES,   'et_features.pkl')

summary = {
    'best_model': best_name,
    'test_R2':    round(best_r2, 4),
    'test_RMSE':  round(best_rmse, 4),
    'features':   FEATURES,
    'individual_models': {k: {m: round(v,4) for m,v in vd.items()} for k,vd in results.items()}
}
with open('model_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*60)
print(f"✅ Model saved: et_ensemble_model.pkl")
print(f"   Best Model : {best_name}")
print(f"   R² Score   : {best_r2:.4f}")
print(f"   RMSE       : {best_rmse:.4f} mm/day")
print("="*60)


✅ Model saved: et_ensemble_model.pkl
   Best Model : Voting Ensemble
   R² Score   : 0.1559
   RMSE       : 250.9354 mm/day
